# Cleanup - Retire Your Workshop

**Draft - not tested. Destructive operations below are opt-in.** Closing a notebook does not remove cloud resources or revoke keys. Review ownership before deleting anything. Attach-existing resources must not be removed wholesale.

Keep a sanitized explanation of your findings if desired. Do not share keys, `.env`, full traces, or customer data. This workshop uses only synthetic records.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
resource_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(resource_root))
from workshop import configure, required, workshop_id, ROOT, source_names, delete_owned_sources, foundry_clients
configure()
print('Exact source indices eligible for ownership-checked deletion:')
display(source_names(include_safety=True))

## Remove native context resources deliberately

In the intended Kibana space, inspect your recorded AI Index and workflow IDs. Disable scheduled automations before removing their source data. Remove only your workshop automations, registered AI Index, and its backing resources as appropriate for the native UI. Do not delete broad `ai-index-*` patterns.

Native API deletion/backing-data semantics are not established by this draft, so these operations are deliberately not automated. Record what remains and any continuing inference charges.

In [ ]:
DELETE_SOURCE_INDICES = False
if DELETE_SOURCE_INDICES:
    confirmation = input(f'Type DELETE {workshop_id()} to remove only owned source/fixture indices: ')
    display(delete_owned_sources(confirmation))
else:
    print('Source deletion skipped.')

## Review Foundry resources

The local agent manifests identify the versions created by the notebooks. Inspect the project, workshop ID, name, and version before deleting. Re-running creation may leave earlier versions: review those in the portal as well.

Also review conversations/responses in exported runs, evaluations and their runs, datasets if uploaded, and both MCP connections. Agent deletion alone does not clean up those artifacts. Use the portal's supported controls or an administrator-approved retention process.

In [ ]:
manifests = [json.loads(path.read_text()) for path in (ROOT / 'outputs').glob('*-agent.json')]
display(manifests)
DELETE_AGENT_VERSIONS = False
if DELETE_AGENT_VERSIONS:
    if input(f'Type DELETE AGENTS {workshop_id()}: ') != f'DELETE AGENTS {workshop_id()}':
        raise ValueError('Agent deletion not confirmed.')
    project, client = foundry_clients()
    expected_endpoint = required('AZURE_AI_PROJECT_ENDPOINT').rstrip('/')
    for manifest in manifests:
        if manifest['workshop_id'] != workshop_id() or manifest['project_endpoint'] != expected_endpoint:
            raise ValueError('Agent manifest belongs to another workshop/project.')
        if not manifest['name'].endswith('-' + workshop_id()):
            raise ValueError('Unexpected agent name in manifest.')
        project.agents.delete_version(agent_name=manifest['name'], agent_version=manifest['version'])
        print('Deleted version:', manifest['name'], manifest['version'])

## Optional: remove the dedicated Azure resource group

Only use this for the create-new path, after checking every contained resource. The name and tag check are safeguards, not permission to delete unrelated resources someone later added. The active subscription is shown for review. Do not delete an existing shared Foundry project or its resource group.

In [ ]:
DELETE_DEDICATED_AZURE_GROUP = False
if DELETE_DEDICATED_AZURE_GROUP:
    group = f'rg-context-quest-{workshop_id()}'
    subprocess.run(['az', 'account', 'show', '--query', '{name:name,id:id}', '-o', 'table'], check=True)
    details = json.loads(subprocess.check_output(['az', 'group', 'show', '--name', group, '-o', 'json'], text=True))
    if (details.get('tags') or {}).get('workshop') != workshop_id():
        raise ValueError('Refusing an unowned resource group.')
    subprocess.run(['az', 'resource', 'list', '--resource-group', group, '--query', '[].{name:name,type:type}', '-o', 'table'], check=True)
    if input(f'Review ALL resources above. Type the exact group name {group}: ') != group:
        raise ValueError('Resource-group deletion not confirmed.')
    subprocess.run(['az', 'group', 'delete', '--name', group, '--yes'], check=True)
else:
    print('Azure resource-group deletion skipped.')

## Final ownership check

Revoke workshop setup/runtime API keys, remove obsolete Foundry connections, and inspect remaining Elastic/Foundry resources and charges. Stop or delete the Codespace from GitHub. Keep local evidence only as required; remove sensitive outputs deliberately rather than using a broad recursive cleanup command.